In [ ]:
!pip install -q "transformers==4.57.1" accelerate bitsandbytes sacrebleu nltk torch pandas

In [ ]:
#!/usr/bin/env python3
# Kaggle setup (run in a cell first):
# !pip install -q "transformers==4.46.3" accelerate bitsandbytes sacrebleu nltk torch pandas
# Add HealthParaphrasing_stratified_200.csv as a Kaggle dataset input
# IMPORTANT: transformers>=4.5x uses a new threaded weight loader that
# materializes several tensors in full precision before quantizing them,
# causing OOM spikes during load even when the quantized model would fit.
# Pinning to 4.46.3 restores the sequential shard-by-shard loader.

import os
import time
import logging
import warnings
import random
import torch
import pandas as pd
import nltk
import sacrebleu
from nltk.translate.meteor_score import meteor_score as meteor_fn
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", message=".*generation flags.*")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

CSV_PATH = "/kaggle/input/datasets/syedmdnafissameen/pera2hel/HealthParaphrasing_stratified_200.csv"
MODEL_ID = "md-nishat-008/TigerLLM-9B-it"

OUTPUT_DIR = "/kaggle/working/bangla_paraphrase_benchmark_tigerllm_fewshot"

NUM_SAMPLES = None

NUM_SHOTS = 5
SHOT_SEED = 42

BATCH_SIZE = 10
MAX_NEW_TOKENS = 256
MAX_INPUT_LENGTH = 2048

SYSTEM_PROMPT = """You are a Bengali paraphrase generation model.

Given a Bengali sentence, generate a single paraphrased version of it.

Rules:
- Preserve the original meaning exactly.
- Vary the vocabulary and sentence structure.
- Output ONLY the paraphrased Bengali sentence.
- Do not include explanations, labels, quotation marks, or extra text.

You will first be shown some Bengali paraphrasing examples.
Follow the same task for the final sentence."""


def build_user_message(sentence: str) -> str:
    return f"Paraphrase this Bengali sentence:\n{sentence}"


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def _rouge_n(pred_tokens, ref_tokens, n):
    pred_ng = _ngrams(pred_tokens, n)
    ref_ng = _ngrams(ref_tokens, n)
    overlap = sum((pred_ng & ref_ng).values())
    p = overlap / max(sum(pred_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()), 1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f


def _lcs_len(a, b):
    m, n = len(a), len(b)
    previous = [0] * (n + 1)
    for i in range(1, m + 1):
        current = [0] * (n + 1)
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                current[j] = previous[j - 1] + 1
            else:
                current[j] = max(previous[j], current[j - 1])
        previous = current
    return previous[n]


def _rouge_l(pred_tokens, ref_tokens):
    lcs = _lcs_len(pred_tokens, ref_tokens)
    p = lcs / max(len(pred_tokens), 1)
    r = lcs / max(len(ref_tokens), 1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f


def load_model():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No GPU detected. In Colab: Runtime > Change runtime type > GPU."
        )

    import gc
    gc.collect()
    torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        max_memory={0: "11GiB", 1: "11GiB", "cpu": "40GiB"},
        trust_remote_code=True,
    )

    model.eval()

    return model, tokenizer


def choose_few_shot_examples(df):
    valid_indices = []
    for i in range(len(df)):
        source = clean_text(df.iloc[i]["source_sentence"])
        reference = clean_text(df.iloc[i]["paraphrased_sentence"])
        if source and reference:
            valid_indices.append(i)

    if len(valid_indices) <= NUM_SHOTS:
        raise ValueError(f"Dataset must contain more than {NUM_SHOTS} valid rows.")

    rng = random.Random(SHOT_SEED)
    shot_indices = rng.sample(valid_indices, NUM_SHOTS)

    examples = []
    for idx in shot_indices:
        examples.append({
            "index": idx,
            "source": clean_text(df.iloc[idx]["source_sentence"]),
            "reference": clean_text(df.iloc[idx]["paraphrased_sentence"]),
        })

    return examples, shot_indices


def build_prompt(tokenizer, source_sentence, few_shot_examples):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for example in few_shot_examples:
        messages.append({"role": "user", "content": build_user_message(example["source"])})
        messages.append({"role": "assistant", "content": example["reference"]})

    messages.append({"role": "user", "content": build_user_message(source_sentence)})

    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        parts = [SYSTEM_PROMPT]
        for i, example in enumerate(few_shot_examples, start=1):
            parts.append(f"Example {i}\n\nInput:\n{example['source']}\n\nOutput:\n{example['reference']}")
        parts.append(f"Now complete the following task.\n\nInput:\n{source_sentence}\n\nOutput:")
        prompt = "\n\n".join(parts)

    return prompt


def checkpoint_path(out_dir, model_id):
    safe_model_id = model_id.replace("/", "__")
    return os.path.join(out_dir, f"checkpoint_{safe_model_id}_{NUM_SHOTS}shot.csv")


def load_checkpoint(out_dir, model_id):
    path = checkpoint_path(out_dir, model_id)
    if os.path.exists(path):
        return pd.read_csv(path)
    return None


def save_checkpoint(out_dir, model_id, rows):
    path = checkpoint_path(out_dir, model_id)
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")


def clean_prediction(text):
    text = str(text).strip()

    prefixes = [
        "Output:", "Paraphrase:", "Paraphrased sentence:",
        "Paraphrased Bengali sentence:", "উত্তর:", "প্যারাফ্রেজ:",
    ]

    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    if "\n" in text:
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        if lines:
            text = lines[0]

    text = text.strip()

    if len(text) >= 2 and text[0] in ['"', "'", "\u201c", "\u2018"] and text[-1] in ['"', "'", "\u201d", "\u2019"]:
        text = text[1:-1].strip()

    return text


@torch.inference_mode()
def run_batch(model, tokenizer, batch_rows, few_shot_examples):
    prompts = [
        build_prompt(tokenizer, str(r["source"]), few_shot_examples)
        for r in batch_rows
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.monotonic()

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.monotonic() - start
    per_item_latency = elapsed / len(batch_rows)

    padded_input_length = inputs["input_ids"].shape[1]
    generated_ids = [output_ids[i, padded_input_length:] for i in range(len(batch_rows))]

    decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    results = []
    for r, text in zip(batch_rows, decoded):
        pred = clean_prediction(text)
        results.append({
            "dataset_index": r["dataset_index"],
            "source": r["source"],
            "reference": r["reference"],
            "prediction": pred,
            "latency": per_item_latency,
            "error": None if pred else "empty_output",
        })

    return results


def run_model_on_dataset(model, tokenizer, eval_df, out_dir, model_id, few_shot_examples):
    existing = load_checkpoint(out_dir, model_id)
    rows_done = existing.to_dict("records") if existing is not None else []

    completed_indices = set()
    for row in rows_done:
        try:
            completed_indices.add(int(row["dataset_index"]))
        except Exception:
            pass

    remaining_rows = []
    for _, row in eval_df.iterrows():
        dataset_index = int(row["dataset_index"])
        if dataset_index not in completed_indices:
            remaining_rows.append({
                "dataset_index": dataset_index,
                "source": clean_text(row["source_sentence"]),
                "reference": clean_text(row["paraphrased_sentence"]),
            })

    if not remaining_rows:
        print("Already complete — loaded from checkpoint.")
        return rows_done

    total_batches = (len(remaining_rows) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_number in range(total_batches):
        start_idx = batch_number * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(remaining_rows))
        batch_rows = remaining_rows[start_idx:end_idx]

        print(f"\nBatch {batch_number + 1}/{total_batches} — dataset rows "
              f"{batch_rows[0]['dataset_index']}..{batch_rows[-1]['dataset_index']}")

        try:
            batch_results = run_batch(model, tokenizer, batch_rows, few_shot_examples)
        except torch.cuda.OutOfMemoryError as e:
            print(f"CUDA OOM: {e}")
            torch.cuda.empty_cache()
            batch_results = []
            for row in batch_rows:
                try:
                    single_result = run_batch(model, tokenizer, [row], few_shot_examples)
                    batch_results.extend(single_result)
                except Exception as single_e:
                    batch_results.append({
                        **row, "prediction": "", "latency": None,
                        "error": f"single_error:{single_e}",
                    })
        except Exception as e:
            print(f"Batch failed: {e}")
            batch_results = [
                {**r, "prediction": "", "latency": None, "error": f"batch_error:{e}"}
                for r in batch_rows
            ]

        rows_done.extend(batch_results)
        rows_done = sorted(rows_done, key=lambda x: int(x["dataset_index"]))
        save_checkpoint(out_dir, model_id, rows_done)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return rows_done


def compute_metrics(rows):
    predictions, references, latencies = [], [], []

    for r in rows:
        pred = clean_text(r.get("prediction", ""))
        ref = clean_text(r.get("reference", ""))
        err = r.get("error")
        valid_error = err is None or str(err).strip() == "" or str(err).lower() == "nan"

        if pred and ref and valid_error:
            predictions.append(pred)
            references.append(ref)

        latency = r.get("latency")
        if latency is not None and str(latency).lower() != "nan":
            try:
                latencies.append(float(latency))
            except Exception:
                pass

    if not predictions:
        print("No valid predictions found.")
        return {}

    print(f"\nEvaluating {len(predictions)} valid predictions...")

    bleu_score = sacrebleu.corpus_bleu(predictions, [references], tokenize="none").score / 100

    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        pred_tokens = pred.split()
        ref_tokens = ref.split()
        r1.append(_rouge_n(pred_tokens, ref_tokens, 1))
        r2.append(_rouge_n(pred_tokens, ref_tokens, 2))
        rl.append(_rouge_l(pred_tokens, ref_tokens))

    meteor_scores = []
    for pred, ref in zip(predictions, references):
        try:
            score = meteor_fn([ref.split()], pred.split())
        except Exception:
            score = 0.0
        meteor_scores.append(score)

    return {
        "BLEU": bleu_score,
        "ROUGE-1 (F1)": sum(r1) / len(r1),
        "ROUGE-2 (F1)": sum(r2) / len(r2),
        "ROUGE-L (F1)": sum(rl) / len(rl),
        "METEOR": sum(meteor_scores) / len(meteor_scores),
        "total": len(predictions),
        "avg_latency": sum(latencies) / len(latencies) if latencies else None,
    }


def save_few_shot_examples(few_shot_examples, out_dir):
    examples_df = pd.DataFrame(few_shot_examples)
    examples_df.to_csv(os.path.join(out_dir, "few_shot_examples.csv"), index=False, encoding="utf-8-sig")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)

    required_columns = {"source_sentence", "paraphrased_sentence"}
    if not required_columns.issubset(df.columns):
        raise ValueError("CSV must contain 'source_sentence' and 'paraphrased_sentence' columns")

    df = df.dropna(subset=["source_sentence", "paraphrased_sentence"]).reset_index(drop=True)
    df["source_sentence"] = df["source_sentence"].astype(str).str.strip()
    df["paraphrased_sentence"] = df["paraphrased_sentence"].astype(str).str.strip()
    df = df[(df["source_sentence"] != "") & (df["paraphrased_sentence"] != "")].reset_index(drop=True)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"\nFull dataset: {len(df)} sentence pairs")

    few_shot_examples, shot_indices = choose_few_shot_examples(df)
    save_few_shot_examples(few_shot_examples, OUTPUT_DIR)

    print(f"\nFew-shot setting: {NUM_SHOTS}-shot")
    print(f"Few-shot example indices: {shot_indices}")
    print("\n── Few-shot examples ──")
    for i, example in enumerate(few_shot_examples, start=1):
        print(f"\nExample {i}")
        print(f"Source:    {example['source']}")
        print(f"Reference: {example['reference']}")

    eval_df = df.drop(index=shot_indices).copy()
    eval_df["dataset_index"] = eval_df.index
    eval_df = eval_df.reset_index(drop=True)

    print(f"\nEvaluation dataset: {len(eval_df)} sentence pairs")
    print(f"Demonstration examples excluded from evaluation: {NUM_SHOTS}")

    model, tokenizer = load_model()

    rows = run_model_on_dataset(
        model=model,
        tokenizer=tokenizer,
        eval_df=eval_df,
        out_dir=OUTPUT_DIR,
        model_id=MODEL_ID,
        few_shot_examples=few_shot_examples,
    )

    metrics = compute_metrics(rows)

    predictions_path = os.path.join(OUTPUT_DIR, "predictions_fewshot.csv")
    pd.DataFrame(rows).to_csv(predictions_path, index=False, encoding="utf-8-sig")

    if not metrics:
        raise RuntimeError("No metrics could be computed.")

    results_df = pd.DataFrame([{
        "model": MODEL_ID,
        "prompt_type": "few-shot",
        "num_shots": NUM_SHOTS,
        "shot_seed": SHOT_SEED,
        "BLEU": round(metrics["BLEU"], 4),
        "ROUGE-1 (F1)": round(metrics["ROUGE-1 (F1)"], 4),
        "ROUGE-2 (F1)": round(metrics["ROUGE-2 (F1)"], 4),
        "ROUGE-L (F1)": round(metrics["ROUGE-L (F1)"], 4),
        "METEOR": round(metrics["METEOR"], 4),
        "total_sentences": metrics["total"],
        "avg_latency_s": metrics["avg_latency"],
    }])

    results_path = os.path.join(OUTPUT_DIR, "benchmark_results_fewshot.csv")
    results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

    print("\n── Few-Shot Evaluation Results ──")
    for metric in ["BLEU", "ROUGE-1 (F1)", "ROUGE-2 (F1)", "ROUGE-L (F1)", "METEOR"]:
        print(f"  {metric:<15} {metrics[metric]:.4f}")

    print(f"\n  Few-shot:        {NUM_SHOTS}-shot")
    print(f"  Total evaluated: {metrics['total']} sentences")

    if metrics["avg_latency"] is not None:
        print(f"  Avg latency:     {metrics['avg_latency']:.2f}s")

    print(f"\n  Predictions: {predictions_path}")
    print(f"  Results:     {results_path}")


if __name__ == "__main__":
    main()